<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4K0A_Frozen_Threshold_Bootstrap_Support_Materialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES Stage 6C — Cell 6C-4K0A
## Frozen 0.50-Threshold Bootstrap Support Materialization

Run the single code cell below before rerunning the final integrated Cell 6C-4K0 V2 notebook.


In [1]:
# ==================================================================================================
# GES STAGE 6C — CELL 6C-4K0A
# FROZEN 0.50-THRESHOLD BOOTSTRAP SUPPORT MATERIALIZATION
#
# Purpose
# -------
# Cell 6C-2C2 completed the prespecified paired 2,000-replicate threshold bootstrap in memory,
# but intentionally wrote no scientific artifact. The final integrated Stage 6C freeze therefore
# cannot truthfully claim checksum-protected threshold-bootstrap support until those already
# completed results are independently reconstructed, serialized, checksum-protected, read back,
# and matched to the historical output.
#
# Scientific boundary
# -------------------
# - Uses the immutable Stage 6B 66,636-row evaluable cohort.
# - Uses the unchanged inherited threshold: P(stable) < 0.50 predicts instability, equivalently
#   frozen instability risk > 0.50.
# - Uses 2,000 paired ordinary row-bootstrap replicates, NumPy default_rng seed 42, batch size 25.
# - Uses identical row draws for Full GES and No-star GES.
# - Performs no threshold selection, optimization, recalibration, model fitting, score fitting,
#   outcome revision, linkage revision, gene reassignment, or Experiment 2 work.
# ==================================================================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

NOTEBOOK_FILENAME = (
    "GES_Stage6C_Cell_6C_4K0A_Frozen_Threshold_Bootstrap_"
    "Support_Materialization.ipynb"
)
print(f"Use this Colab notebook file name: {NOTEBOOK_FILENAME}")

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. IMMUTABLE INPUTS, EXPECTED IDENTITIES, AND VERSIONED OUTPUTS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

PROJECT_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"
STAGE6_DIR = PROJECT_DIR / "data_processed" / "stage6_temporal_validation"

EVALUABLE_PARQUET = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")
EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

PRIOR_MANIFEST = (
    PROJECT_DIR
    / "configs"
    / "stage6_temporal_validation"
    / "stage6c_4j0_leave_one_gene_out_validation_materialization_v1"
    / "stage6c_4j0_leave_one_gene_out_validation_manifest_v1.json"
)
PRIOR_MANIFEST_SIDECAR = Path(str(PRIOR_MANIFEST) + ".sha256")
EXPECTED_PRIOR_MANIFEST_SHA256 = (
    "05c63d4a4a2ba897e73f56d91073903f069c56ff88fb648106aefa00bcf5b8cf"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

KEY_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
OUTCOME_COLUMN = "primary_future_instability"
FULL_RISK_COLUMN = "full_ges_instability_risk_t0"
NO_STAR_RISK_COLUMN = "no_star_ges_instability_risk_t0"

FROZEN_THRESHOLD = 0.50
BOOTSTRAP_REPLICATES = 2_000
BOOTSTRAP_SEED = 42
BOOTSTRAP_BATCH_SIZE = 25
THRESHOLD_RULE = "P(stable) < 0.50 predicts instability"

PACKAGE_VERSION = "v1"
PACKAGE_NAME = "stage6c_4k0a_frozen_threshold_bootstrap_support_materialization_v1"

TABLE_DIR = (
    PROJECT_DIR
    / "outputs"
    / "tables"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)
QC_DIR = (
    PROJECT_DIR
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)
CONFIG_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6_temporal_validation"
    / PACKAGE_NAME
)

for directory in [TABLE_DIR, QC_DIR, CONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PATHS = {
    "design": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_design_v1.json",
    "point_estimates": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_point_estimates_v1.csv",
    "replicates": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_replicates_v1.parquet",
    "model_intervals": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_model_intervals_v1.csv",
    "paired_comparisons": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_paired_comparisons_v1.csv",
    "count_qc": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_count_qc_v1.csv",
    "historical_concordance": TABLE_DIR / "stage6c_frozen_threshold_bootstrap_historical_concordance_v1.csv",
    "qc": QC_DIR / "stage6c_frozen_threshold_bootstrap_materialization_qc_v1.json",
    "manifest": CONFIG_DIR / "stage6c_frozen_threshold_bootstrap_support_manifest_v1.json",
}

CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + ".sha256")


def sidecar_ok(path: Path) -> bool:
    sc = sidecar_path(path)
    return path.exists() and sc.exists() and read_sidecar_hash(sc) == sha256_file(path)


def write_sidecar(path: Path) -> Path:
    digest = sha256_file(path)
    sc = sidecar_path(path)
    sc.write_text(f"{digest}  {path.name}\n", encoding="utf-8")
    return sc


def native(value):
    if isinstance(value, dict):
        return {str(key): native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]
    if isinstance(value, np.ndarray):
        return [native(item) for item in value.tolist()]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def write_json(path: Path, payload) -> None:
    path.write_text(
        json.dumps(native(payload), indent=2, sort_keys=True, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


def write_csv(path: Path, frame: pd.DataFrame) -> None:
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.12g")


def write_parquet(path: Path, frame: pd.DataFrame) -> None:
    frame.to_parquet(path, index=False, engine="pyarrow", compression="snappy")


def validate_binary(series: pd.Series, label: str) -> np.ndarray:
    values = pd.to_numeric(series, errors="raise")
    if values.isna().any():
        raise AssertionError(f"{label} contains missing values.")
    unique = set(values.unique().tolist())
    if not unique.issubset({0, 1, 0.0, 1.0, False, True}):
        raise AssertionError(f"{label} is not binary: {sorted(unique)}")
    return values.to_numpy(dtype=np.int8)


def validate_risk(series: pd.Series, label: str) -> np.ndarray:
    values = pd.to_numeric(series, errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise AssertionError(f"{label} contains missing or nonfinite values.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"{label} contains values outside [0,1].")
    if len(np.unique(values)) < 2:
        raise AssertionError(f"{label} has insufficient variation.")
    return values


def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    output = np.full(np.broadcast_shapes(numerator.shape, denominator.shape), np.nan, dtype=float)
    return np.divide(numerator, denominator, out=output, where=denominator != 0)


def calculate_metric_arrays(tp, fp, tn, fn) -> OrderedDict:
    tp = np.asarray(tp, dtype=float)
    fp = np.asarray(fp, dtype=float)
    tn = np.asarray(tn, dtype=float)
    fn = np.asarray(fn, dtype=float)

    total = tp + fp + tn + fn
    events = tp + fn
    negatives = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    sensitivity = safe_divide(tp, events)
    specificity = safe_divide(tn, negatives)
    ppv = safe_divide(tp, predicted_positive)
    npv = safe_divide(tn, predicted_negative)
    accuracy = safe_divide(tp + tn, total)
    balanced_accuracy = (sensitivity + specificity) / 2.0
    f1_score = safe_divide(2.0 * tp, 2.0 * tp + fp + fn)

    mcc_denominator = np.sqrt(
        predicted_positive * events * negatives * predicted_negative
    )
    mcc = safe_divide(tp * tn - fp * fn, mcc_denominator)

    predicted_positive_fraction = safe_divide(predicted_positive, total)
    predicted_negative_fraction = safe_divide(predicted_negative, total)
    predicted_positive_event_rate = ppv
    predicted_negative_event_rate = safe_divide(fn, predicted_negative)
    event_rate_difference = predicted_positive_event_rate - predicted_negative_event_rate
    relative_event_risk = safe_divide(
        predicted_positive_event_rate,
        predicted_negative_event_rate,
    )
    prevalence = safe_divide(events, total)
    enrichment_vs_overall_prevalence = safe_divide(
        predicted_positive_event_rate,
        prevalence,
    )
    event_capture_fraction = sensitivity
    youden_j = sensitivity + specificity - 1.0

    return OrderedDict(
        [
            ("sensitivity", sensitivity),
            ("specificity", specificity),
            ("positive_predictive_value", ppv),
            ("negative_predictive_value", npv),
            ("accuracy", accuracy),
            ("balanced_accuracy", balanced_accuracy),
            ("f1_score", f1_score),
            ("matthews_correlation", mcc),
            ("predicted_positive_fraction", predicted_positive_fraction),
            ("predicted_negative_fraction", predicted_negative_fraction),
            ("predicted_positive_event_rate", predicted_positive_event_rate),
            ("predicted_negative_event_rate", predicted_negative_event_rate),
            ("event_rate_difference", event_rate_difference),
            ("relative_event_risk", relative_event_risk),
            ("enrichment_vs_overall_prevalence", enrichment_vs_overall_prevalence),
            ("event_capture_fraction", event_capture_fraction),
            ("youden_j", youden_j),
        ]
    )


def scalar_metrics(tp: int, fp: int, tn: int, fn: int) -> OrderedDict:
    arrays = calculate_metric_arrays(
        np.asarray([tp]),
        np.asarray([fp]),
        np.asarray([tn]),
        np.asarray([fn]),
    )
    return OrderedDict((key, float(value[0])) for key, value in arrays.items())


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC, STRUCTURAL, AND LINEAGE PREFLIGHT
# --------------------------------------------------------------------------------------------------

for required in [
    EVALUABLE_PARQUET,
    EVALUABLE_SIDECAR,
    PRIOR_MANIFEST,
    PRIOR_MANIFEST_SIDECAR,
]:
    if not required.exists():
        raise FileNotFoundError(f"Required frozen artifact is missing:\n{required}")

if sha256_file(EVALUABLE_PARQUET) != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError("Stage 6B evaluable cohort SHA-256 mismatch.")
if not sidecar_ok(EVALUABLE_PARQUET):
    raise AssertionError("Stage 6B evaluable cohort sidecar verification failed.")

if sha256_file(PRIOR_MANIFEST) != EXPECTED_PRIOR_MANIFEST_SHA256:
    raise AssertionError("Cell 6C-4J0 manifest SHA-256 mismatch.")
if not sidecar_ok(PRIOR_MANIFEST):
    raise AssertionError("Cell 6C-4J0 manifest sidecar verification failed.")

prior_payload = json.loads(PRIOR_MANIFEST.read_text(encoding="utf-8"))
prior_identity = prior_payload.get("cell_id") or prior_payload.get("cell")
if prior_identity != "6C-4J0":
    raise AssertionError(f"Unexpected prior manifest identity: {prior_identity}")
prior_decision = str(prior_payload.get("decision", ""))
if "PASS_STAGE6C" not in prior_decision or "LEAVE_ONE_GENE_OUT" not in prior_decision:
    raise AssertionError("Cell 6C-4J0 manifest does not contain the expected passing decision.")

metadata = pq.ParquetFile(EVALUABLE_PARQUET).metadata
if (metadata.num_rows, metadata.num_columns) != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise AssertionError(
        "Unexpected Stage 6B dimensions: "
        f"{metadata.num_rows:,} × {metadata.num_columns}"
    )

required_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
    FULL_RISK_COLUMN,
    NO_STAR_RISK_COLUMN,
]
missing = [column for column in required_columns if column not in pq.ParquetFile(EVALUABLE_PARQUET).schema_arrow.names]
if missing:
    raise KeyError("Missing required Stage 6B columns:\n" + "\n".join(missing))

cohort = pd.read_parquet(EVALUABLE_PARQUET, columns=required_columns).copy()
if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded Stage 6B row count mismatch.")

cohort[KEY_COLUMN] = cohort[KEY_COLUMN].astype(str).str.strip().str.upper()
if cohort[KEY_COLUMN].eq("").any() or cohort[KEY_COLUMN].duplicated().any():
    raise AssertionError("Stage 6B RCV keys are blank or duplicated.")

row_order = pd.to_numeric(cohort[ROW_ORDER_COLUMN], errors="raise").to_numpy(dtype=np.int64)
if len(np.unique(row_order)) != EXPECTED_ROWS or not np.all(np.diff(row_order) > 0):
    raise AssertionError("Frozen t0_row_order is not unique and strictly increasing.")

outcome = validate_binary(cohort[OUTCOME_COLUMN], OUTCOME_COLUMN)
full_risk = validate_risk(cohort[FULL_RISK_COLUMN], FULL_RISK_COLUMN)
no_star_risk = validate_risk(cohort[NO_STAR_RISK_COLUMN], NO_STAR_RISK_COLUMN)

observed_events = int(outcome.sum())
observed_negatives = int((outcome == 0).sum())
if (observed_events, observed_negatives) != (EXPECTED_EVENTS, EXPECTED_NEGATIVES):
    raise AssertionError(
        f"Outcome accounting mismatch: {observed_events:,} events / "
        f"{observed_negatives:,} negatives."
    )

# Strict inequality is the frozen rule: risk > 0.50, equivalent to P(stable) < 0.50.
full_predicted_instability = full_risk > FROZEN_THRESHOLD
no_star_predicted_instability = no_star_risk > FROZEN_THRESHOLD

MODEL_SPECS = OrderedDict(
    [
        (
            "full_ges",
            {
                "display": "Full GES",
                "prediction": full_predicted_instability,
                "expected_counts": {"tp": 442, "fp": 4057, "tn": 56094, "fn": 6043},
            },
        ),
        (
            "no_star_ges",
            {
                "display": "No-star GES",
                "prediction": no_star_predicted_instability,
                "expected_counts": {"tp": 365, "fp": 1931, "tn": 58220, "fn": 6120},
            },
        ),
    ]
)

point_counts = {}
for model_key, spec in MODEL_SPECS.items():
    prediction = np.asarray(spec["prediction"], dtype=bool)
    tp = int(np.count_nonzero(prediction & (outcome == 1)))
    fp = int(np.count_nonzero(prediction & (outcome == 0)))
    tn = int(np.count_nonzero((~prediction) & (outcome == 0)))
    fn = int(np.count_nonzero((~prediction) & (outcome == 1)))
    observed = {"tp": tp, "fp": fp, "tn": tn, "fn": fn}
    if observed != spec["expected_counts"]:
        raise AssertionError(
            f"Frozen threshold confusion counts changed for {spec['display']}: "
            f"expected {spec['expected_counts']}, observed {observed}"
        )
    point_counts[model_key] = observed


# --------------------------------------------------------------------------------------------------
# 4. REPRODUCE THE ORIGINAL PAIRED 2,000-REPLICATE ROW BOOTSTRAP
# --------------------------------------------------------------------------------------------------

print(
    f"\nReconstructing frozen threshold bootstrap across {EXPECTED_ROWS:,} rows: "
    f"{EXPECTED_EVENTS:,} events and {EXPECTED_NEGATIVES:,} negatives"
)

bootstrap_total_events = np.empty(BOOTSTRAP_REPLICATES, dtype=np.int64)
bootstrap_counts = {
    model_key: {
        "true_positive": np.empty(BOOTSTRAP_REPLICATES, dtype=np.int64),
        "false_positive": np.empty(BOOTSTRAP_REPLICATES, dtype=np.int64),
        "true_negative": np.empty(BOOTSTRAP_REPLICATES, dtype=np.int64),
        "false_negative": np.empty(BOOTSTRAP_REPLICATES, dtype=np.int64),
    }
    for model_key in MODEL_SPECS
}

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_start = time.time()

for batch_start in range(0, BOOTSTRAP_REPLICATES, BOOTSTRAP_BATCH_SIZE):
    batch_stop = min(batch_start + BOOTSTRAP_BATCH_SIZE, BOOTSTRAP_REPLICATES)
    batch_size = batch_stop - batch_start

    sampled_indices = rng.integers(
        0,
        EXPECTED_ROWS,
        size=(batch_size, EXPECTED_ROWS),
    )
    sampled_outcome = outcome[sampled_indices]
    batch_events = sampled_outcome.sum(axis=1, dtype=np.int64)
    bootstrap_total_events[batch_start:batch_stop] = batch_events

    for model_key, spec in MODEL_SPECS.items():
        sampled_prediction = np.asarray(spec["prediction"], dtype=bool)[sampled_indices]
        tp = np.count_nonzero(sampled_prediction & (sampled_outcome == 1), axis=1)
        fp = np.count_nonzero(sampled_prediction & (sampled_outcome == 0), axis=1)
        fn = batch_events - tp
        tn = (EXPECTED_ROWS - batch_events) - fp

        bootstrap_counts[model_key]["true_positive"][batch_start:batch_stop] = tp
        bootstrap_counts[model_key]["false_positive"][batch_start:batch_stop] = fp
        bootstrap_counts[model_key]["true_negative"][batch_start:batch_stop] = tn
        bootstrap_counts[model_key]["false_negative"][batch_start:batch_stop] = fn

    if batch_stop % 250 == 0 or batch_stop == BOOTSTRAP_REPLICATES:
        elapsed = time.time() - bootstrap_start
        print(
            f"  Completed {batch_stop:,}/{BOOTSTRAP_REPLICATES:,} paired replicates "
            f"({elapsed:.1f}s elapsed)"
        )

bootstrap_elapsed_seconds = time.time() - bootstrap_start

# Exact historical stream anchors from the original Cell 6C-2C2 output.
expected_first_ten = {
    "bootstrap_total_events": [6395, 6513, 6595, 6606, 6490, 6486, 6422, 6436, 6609, 6456],
    "full_ges_true_positive": [426, 426, 462, 433, 474, 470, 415, 449, 462, 445],
    "full_ges_false_positive": [4129, 4079, 4062, 4009, 4123, 3860, 3953, 4082, 4147, 4078],
    "full_ges_true_negative": [56112, 56044, 55979, 56021, 56023, 56290, 56261, 56118, 55880, 56102],
    "full_ges_false_negative": [5969, 6087, 6133, 6173, 6016, 6016, 6007, 5987, 6147, 6011],
}
observed_first_ten = {
    "bootstrap_total_events": bootstrap_total_events[:10].tolist(),
    "full_ges_true_positive": bootstrap_counts["full_ges"]["true_positive"][:10].tolist(),
    "full_ges_false_positive": bootstrap_counts["full_ges"]["false_positive"][:10].tolist(),
    "full_ges_true_negative": bootstrap_counts["full_ges"]["true_negative"][:10].tolist(),
    "full_ges_false_negative": bootstrap_counts["full_ges"]["false_negative"][:10].tolist(),
}
if observed_first_ten != expected_first_ten:
    raise RuntimeError(
        "Bootstrap RNG/resampling stream does not match the original Cell 6C-2C2 output.\n"
        + json.dumps(
            {"expected": expected_first_ten, "observed": observed_first_ten},
            indent=2,
        )
    )

bootstrap_metrics = {}
for model_key in MODEL_SPECS:
    counts = bootstrap_counts[model_key]
    bootstrap_metrics[model_key] = calculate_metric_arrays(
        counts["true_positive"],
        counts["false_positive"],
        counts["true_negative"],
        counts["false_negative"],
    )
    for metric_name, values in bootstrap_metrics[model_key].items():
        if values.shape != (BOOTSTRAP_REPLICATES,) or not np.isfinite(values).all():
            raise RuntimeError(
                f"Invalid bootstrap metric array for {model_key}/{metric_name}."
            )


# --------------------------------------------------------------------------------------------------
# 5. MATERIALIZED TABLES
# --------------------------------------------------------------------------------------------------

point_rows = []
point_metrics = {}
for model_key, spec in MODEL_SPECS.items():
    counts = point_counts[model_key]
    metrics = scalar_metrics(
        counts["tp"], counts["fp"], counts["tn"], counts["fn"]
    )
    point_metrics[model_key] = metrics
    point_rows.append(
        {
            "model_key": model_key,
            "model": spec["display"],
            "threshold_rule": THRESHOLD_RULE,
            "threshold": FROZEN_THRESHOLD,
            "threshold_optimized_on_t1": False,
            "threshold_reselected_per_replicate": False,
            "rows": EXPECTED_ROWS,
            "events": EXPECTED_EVENTS,
            "negatives": EXPECTED_NEGATIVES,
            "true_positive": counts["tp"],
            "false_positive": counts["fp"],
            "true_negative": counts["tn"],
            "false_negative": counts["fn"],
            **metrics,
        }
    )
point_estimates = pd.DataFrame(point_rows)

replicate_data = OrderedDict(
    [
        ("bootstrap_replicate", np.arange(1, BOOTSTRAP_REPLICATES + 1, dtype=np.int64)),
        ("bootstrap_total_events", bootstrap_total_events),
        ("bootstrap_prevalence", bootstrap_total_events / EXPECTED_ROWS),
    ]
)
for model_key in MODEL_SPECS:
    counts = bootstrap_counts[model_key]
    replicate_data[f"{model_key}_true_positive"] = counts["true_positive"]
    replicate_data[f"{model_key}_false_positive"] = counts["false_positive"]
    replicate_data[f"{model_key}_true_negative"] = counts["true_negative"]
    replicate_data[f"{model_key}_false_negative"] = counts["false_negative"]
    for metric_name, values in bootstrap_metrics[model_key].items():
        replicate_data[f"{model_key}_{metric_name}"] = values
bootstrap_replicates = pd.DataFrame(replicate_data)
if bootstrap_replicates.shape != (BOOTSTRAP_REPLICATES, 45):
    raise RuntimeError(
        f"Unexpected raw bootstrap table shape: {bootstrap_replicates.shape}; "
        "expected (2000, 45)."
    )

NULL_VALUES = {
    "balanced_accuracy": 0.5,
    "matthews_correlation": 0.0,
    "event_rate_difference": 0.0,
    "relative_event_risk": 1.0,
    "enrichment_vs_overall_prevalence": 1.0,
    "youden_j": 0.0,
}

summary_rows = []
for model_key, spec in MODEL_SPECS.items():
    for metric_name, values in bootstrap_metrics[model_key].items():
        lower, upper = np.percentile(values, [2.5, 97.5])
        null_value = NULL_VALUES.get(metric_name, np.nan)
        support = (
            float(np.mean(values > null_value))
            if np.isfinite(null_value)
            else np.nan
        )
        summary_rows.append(
            {
                "model_key": model_key,
                "model": spec["display"],
                "metric": metric_name,
                "point_estimate": point_metrics[model_key][metric_name],
                "bootstrap_mean": float(np.mean(values)),
                "bootstrap_standard_error": float(np.std(values, ddof=1)),
                "percentile_95_ci_lower": float(lower),
                "percentile_95_ci_upper": float(upper),
                "null_value": null_value,
                "bootstrap_support_above_null": support,
                "interval_excludes_null_above": bool(
                    np.isfinite(null_value) and lower > null_value
                ),
                "interval_excludes_null_below": bool(
                    np.isfinite(null_value) and upper < null_value
                ),
                "bootstrap_replicates": BOOTSTRAP_REPLICATES,
                "bootstrap_seed": BOOTSTRAP_SEED,
                "paired_resamples": True,
                "threshold_rule": THRESHOLD_RULE,
                "threshold_reselected_per_replicate": False,
                "threshold_optimized_on_t1": False,
            }
        )
model_intervals = pd.DataFrame(summary_rows)
if len(model_intervals) != 34:
    raise RuntimeError("Expected 34 model-metric threshold-bootstrap interval rows.")

PAIRED_METRICS = [
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "negative_predictive_value",
    "balanced_accuracy",
    "f1_score",
    "matthews_correlation",
    "event_rate_difference",
]
paired_rows = []
for metric_name in PAIRED_METRICS:
    differences = (
        bootstrap_metrics["full_ges"][metric_name]
        - bootstrap_metrics["no_star_ges"][metric_name]
    )
    lower, upper = np.percentile(differences, [2.5, 97.5])
    paired_rows.append(
        {
            "comparison": "Full GES minus No-star GES",
            "metric": metric_name,
            "point_difference": (
                point_metrics["full_ges"][metric_name]
                - point_metrics["no_star_ges"][metric_name]
            ),
            "bootstrap_mean_difference": float(np.mean(differences)),
            "bootstrap_standard_error": float(np.std(differences, ddof=1)),
            "percentile_95_ci_lower": float(lower),
            "percentile_95_ci_upper": float(upper),
            "directional_probability_above_zero": float(np.mean(differences > 0.0)),
            "interval_excludes_zero_above": bool(lower > 0.0),
            "interval_excludes_zero_below": bool(upper < 0.0),
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "paired_resamples": True,
            "threshold_rule": THRESHOLD_RULE,
        }
    )
paired_comparisons = pd.DataFrame(paired_rows)
if len(paired_comparisons) != 8:
    raise RuntimeError("Expected eight prespecified paired threshold comparisons.")

count_rows = []
for model_key, spec in MODEL_SPECS.items():
    tp = bootstrap_counts[model_key]["true_positive"]
    predicted_positive = (
        bootstrap_counts[model_key]["true_positive"]
        + bootstrap_counts[model_key]["false_positive"]
    )
    count_rows.append(
        {
            "model_key": model_key,
            "model": spec["display"],
            "locked_true_positive": point_counts[model_key]["tp"],
            "bootstrap_mean_true_positive": float(np.mean(tp)),
            "bootstrap_minimum_true_positive": int(np.min(tp)),
            "bootstrap_maximum_true_positive": int(np.max(tp)),
            "locked_predicted_positive": (
                point_counts[model_key]["tp"] + point_counts[model_key]["fp"]
            ),
            "bootstrap_mean_predicted_positive": float(np.mean(predicted_positive)),
            "bootstrap_minimum_predicted_positive": int(np.min(predicted_positive)),
            "bootstrap_maximum_predicted_positive": int(np.max(predicted_positive)),
        }
    )
count_qc = pd.DataFrame(count_rows)


# --------------------------------------------------------------------------------------------------
# 6. HISTORICAL CONCORDANCE AT THE RECORDED PRECISION
# --------------------------------------------------------------------------------------------------

historical_rows = []

def add_historical(result_id, reproduced, historical, digits, source):
    tolerance = 0.51 * (10.0 ** (-digits))
    historical_rows.append(
        {
            "result_id": result_id,
            "source": source,
            "historical_value": historical,
            "reproduced_value": reproduced,
            "recorded_digits": digits,
            "tolerance": tolerance,
            "absolute_difference": abs(float(reproduced) - float(historical)),
            "value_reproduced_at_recorded_precision": bool(
                abs(float(reproduced) - float(historical)) <= tolerance
            ),
        }
    )

historical_summary = {
    # model_key, metric: point, mean, SE, lower, upper
    ("full_ges", "balanced_accuracy"): (0.50035518, 0.50034872, 0.00165445, 0.49715773, 0.50368131),
    ("full_ges", "f1_score"): (0.08048070, 0.08044822, 0.00361651, 0.07341612, 0.08773790),
    ("full_ges", "matthews_correlation"): (0.00083912, 0.00082412, 0.00390886, -0.00668882, 0.00869271),
    ("full_ges", "negative_predictive_value"): (0.90274716, 0.90275948, 0.00118499, 0.90049260, 0.90510268),
    ("full_ges", "positive_predictive_value"): (0.09824405, 0.09821440, 0.00447078, 0.08962896, 0.10711309),
    ("full_ges", "sensitivity"): (0.06815729, 0.06813462, 0.00314313, 0.06206666, 0.07439629),
    ("full_ges", "specificity"): (0.93255307, 0.93256282, 0.00102850, 0.93058457, 0.93462338),
    ("no_star_ges", "balanced_accuracy"): (0.51209059, 0.51210498, 0.00148433, 0.50924474, 0.51508474),
    ("no_star_ges", "f1_score"): (0.08313404, 0.08314636, 0.00410324, 0.07532676, 0.09124992),
    ("no_star_ges", "matthews_correlation"): (0.03929412, 0.03934593, 0.00478516, 0.03026852, 0.04885265),
    ("no_star_ges", "negative_predictive_value"): (0.90488032, 0.90489618, 0.00115254, 0.90262606, 0.90713589),
    ("no_star_ges", "positive_predictive_value"): (0.15897213, 0.15905596, 0.00769408, 0.14410705, 0.17463375),
    ("no_star_ges", "sensitivity"): (0.05628373, 0.05629202, 0.00287179, 0.05064417, 0.06194068),
    ("no_star_ges", "specificity"): (0.96789746, 0.96791794, 0.00069998, 0.96657055, 0.96925942),
}

summary_lookup = model_intervals.set_index(["model_key", "metric"])
fields = [
    "point_estimate",
    "bootstrap_mean",
    "bootstrap_standard_error",
    "percentile_95_ci_lower",
    "percentile_95_ci_upper",
]
for key, historical_values in historical_summary.items():
    row = summary_lookup.loc[key]
    for field, historical in zip(fields, historical_values):
        add_historical(
            f"model_interval__{key[0]}__{key[1]}__{field}",
            float(row[field]),
            historical,
            8,
            "Original Cell 6C-2C2 controlled output",
        )

historical_paired = {
    # metric: point, mean, SE, lower, upper, directional probability > 0
    "sensitivity": (0.01187355, 0.01184260, 0.00133398, 0.00931424, 0.01457541, 1.000000),
    "specificity": (-0.03534438, -0.03535511, 0.00076443, -0.03686805, -0.03388244, 0.000000),
    "positive_predictive_value": (-0.06072807, -0.06084156, 0.00431185, -0.06957687, -0.05247719, 0.000000),
    "negative_predictive_value": (-0.00213317, -0.00213670, 0.00014573, -0.00241623, -0.00184702, 0.000000),
    "balanced_accuracy": (-0.01173541, -0.01175626, 0.00076403, -0.01321731, -0.01021195, 0.000000),
    "f1_score": (-0.00265334, -0.00269814, 0.00176755, -0.00619267, 0.00090045, 0.063000),
    "matthews_correlation": (-0.03845501, -0.03852181, 0.00227121, -0.04305483, -0.03385368, 0.000000),
    "event_rate_difference": (-0.06286124, -0.06297826, 0.00437904, -0.07189661, -0.05451265, 0.000000),
}

paired_lookup = paired_comparisons.set_index("metric")
paired_fields = [
    "point_difference",
    "bootstrap_mean_difference",
    "bootstrap_standard_error",
    "percentile_95_ci_lower",
    "percentile_95_ci_upper",
    "directional_probability_above_zero",
]
for metric_name, historical_values in historical_paired.items():
    row = paired_lookup.loc[metric_name]
    for field, historical in zip(paired_fields, historical_values):
        digits = 6 if field == "directional_probability_above_zero" else 8
        add_historical(
            f"paired__{metric_name}__{field}",
            float(row[field]),
            historical,
            digits,
            "Original Cell 6C-2C2 controlled output",
        )

historical_count_qc = {
    "full_ges": {
        "bootstrap_mean_true_positive": 441.7945,
        "bootstrap_minimum_true_positive": 379,
        "bootstrap_maximum_true_positive": 506,
        "bootstrap_mean_predicted_positive": 4498.2685,
        "bootstrap_minimum_predicted_positive": 4284,
        "bootstrap_maximum_predicted_positive": 4765,
    },
    "no_star_ges": {
        "bootstrap_mean_true_positive": 365.0065,
        "bootstrap_minimum_true_positive": 305,
        "bootstrap_maximum_true_positive": 432,
        "bootstrap_mean_predicted_positive": 2294.8030,
        "bootstrap_minimum_predicted_positive": 2139,
        "bootstrap_maximum_predicted_positive": 2456,
    },
}
count_lookup = count_qc.set_index("model_key")
for model_key, expected_fields in historical_count_qc.items():
    for field, historical in expected_fields.items():
        digits = 4 if "mean" in field else 0
        add_historical(
            f"count_qc__{model_key}__{field}",
            float(count_lookup.loc[model_key, field]),
            historical,
            digits,
            "Original Cell 6C-2C2 controlled output",
        )

historical_concordance = pd.DataFrame(historical_rows)
if not historical_concordance["value_reproduced_at_recorded_precision"].all():
    failures = historical_concordance.loc[
        ~historical_concordance["value_reproduced_at_recorded_precision"]
    ]
    raise RuntimeError(
        "Historical threshold-bootstrap concordance failed:\n"
        + failures.to_string(index=False)
    )


# --------------------------------------------------------------------------------------------------
# 7. SCIENTIFIC-CONCLUSION AND COMPLETENESS QC
# --------------------------------------------------------------------------------------------------

full_summary = summary_lookup.loc["full_ges"]
no_star_summary = summary_lookup.loc["no_star_ges"]

checks = []

def check(name, passed, details):
    checks.append({"check_name": name, "passed": bool(passed), "details": native(details)})

check("stage6b_hash_and_sidecar", sha256_file(EVALUABLE_PARQUET) == EXPECTED_EVALUABLE_SHA256 and sidecar_ok(EVALUABLE_PARQUET), sha256_file(EVALUABLE_PARQUET))
check("prior_4j_manifest_hash_and_sidecar", sha256_file(PRIOR_MANIFEST) == EXPECTED_PRIOR_MANIFEST_SHA256 and sidecar_ok(PRIOR_MANIFEST), sha256_file(PRIOR_MANIFEST))
check("stage6b_dimensions", (metadata.num_rows, metadata.num_columns) == (EXPECTED_ROWS, EXPECTED_COLUMNS), [metadata.num_rows, metadata.num_columns])
check("outcome_accounting", (observed_events, observed_negatives) == (EXPECTED_EVENTS, EXPECTED_NEGATIVES), [observed_events, observed_negatives])
check("frozen_confusion_counts", all(point_counts[key] == MODEL_SPECS[key]["expected_counts"] for key in MODEL_SPECS), point_counts)
check("bootstrap_attempts", len(bootstrap_replicates) == BOOTSTRAP_REPLICATES, len(bootstrap_replicates))
check("bootstrap_raw_shape", bootstrap_replicates.shape == (2000, 45), bootstrap_replicates.shape)
check("bootstrap_rng_stream_first_ten", observed_first_ten == expected_first_ten, observed_first_ten)
check("model_interval_rows", len(model_intervals) == 34, len(model_intervals))
check("paired_comparison_rows", len(paired_comparisons) == 8, len(paired_comparisons))
check("count_qc_rows", len(count_qc) == 2, len(count_qc))
check("historical_concordance", historical_concordance["value_reproduced_at_recorded_precision"].all(), historical_concordance.loc[~historical_concordance["value_reproduced_at_recorded_precision"]].to_dict("records"))
check(
    "full_ges_threshold_effectively_null",
    float(full_summary.loc["balanced_accuracy", "percentile_95_ci_lower"]) <= 0.5 <= float(full_summary.loc["balanced_accuracy", "percentile_95_ci_upper"])
    and float(full_summary.loc["matthews_correlation", "percentile_95_ci_lower"]) <= 0.0 <= float(full_summary.loc["matthews_correlation", "percentile_95_ci_upper"])
    and float(full_summary.loc["relative_event_risk", "percentile_95_ci_lower"]) <= 1.0 <= float(full_summary.loc["relative_event_risk", "percentile_95_ci_upper"]),
    {
        "balanced_accuracy_ci": full_summary.loc["balanced_accuracy", ["percentile_95_ci_lower", "percentile_95_ci_upper"]].to_dict(),
        "mcc_ci": full_summary.loc["matthews_correlation", ["percentile_95_ci_lower", "percentile_95_ci_upper"]].to_dict(),
        "relative_risk_ci": full_summary.loc["relative_event_risk", ["percentile_95_ci_lower", "percentile_95_ci_upper"]].to_dict(),
    },
)
check(
    "no_star_enriched_but_very_low_recall",
    float(no_star_summary.loc["relative_event_risk", "percentile_95_ci_lower"]) > 1.0
    and float(no_star_summary.loc["balanced_accuracy", "percentile_95_ci_lower"]) > 0.5
    and float(no_star_summary.loc["matthews_correlation", "percentile_95_ci_lower"]) > 0.0
    and float(no_star_summary.loc["sensitivity", "percentile_95_ci_upper"]) < 0.07,
    {
        "relative_risk_ci": no_star_summary.loc["relative_event_risk", ["percentile_95_ci_lower", "percentile_95_ci_upper"]].to_dict(),
        "sensitivity_ci": no_star_summary.loc["sensitivity", ["percentile_95_ci_lower", "percentile_95_ci_upper"]].to_dict(),
    },
)
check(
    "paired_f1_difference_not_supported",
    float(paired_lookup.loc["f1_score", "percentile_95_ci_lower"]) <= 0.0 <= float(paired_lookup.loc["f1_score", "percentile_95_ci_upper"]),
    paired_lookup.loc["f1_score", ["percentile_95_ci_lower", "percentile_95_ci_upper"]].to_dict(),
)
check("threshold_never_optimized", True, {"threshold": FROZEN_THRESHOLD, "selected_using_t1": False})
check("experiment_2_not_started", True, False)

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))


# --------------------------------------------------------------------------------------------------
# 8. VERSIONED WRITES, SIDECARS, QC, MANIFEST, AND FRESH SEMANTIC READBACK
# --------------------------------------------------------------------------------------------------

design_payload = {
    "cell_id": "6C-4K0A",
    "package_name": PACKAGE_NAME,
    "created_utc": CREATED_UTC,
    "purpose": "Independent materialization of the completed in-memory Cell 6C-2C2 frozen-threshold bootstrap",
    "source": {
        "stage6b_evaluable_path": str(EVALUABLE_PARQUET),
        "stage6b_evaluable_sha256": EXPECTED_EVALUABLE_SHA256,
        "prior_4j_manifest_path": str(PRIOR_MANIFEST),
        "prior_4j_manifest_sha256": EXPECTED_PRIOR_MANIFEST_SHA256,
    },
    "cohort": {
        "rows": EXPECTED_ROWS,
        "events": EXPECTED_EVENTS,
        "negatives": EXPECTED_NEGATIVES,
    },
    "threshold": {
        "value": FROZEN_THRESHOLD,
        "rule": THRESHOLD_RULE,
        "equivalent_instability_risk_rule": "frozen instability risk > 0.50",
        "selected_using_t1": False,
        "reselected_per_replicate": False,
        "optimized": False,
    },
    "bootstrap": {
        "method": "paired nonparametric ordinary row bootstrap",
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "numpy_rng": "numpy.random.default_rng",
        "batch_size": BOOTSTRAP_BATCH_SIZE,
        "same_row_samples_for_both_models": True,
        "resampling_unit": "individual frozen evaluable row",
    },
    "models": [spec["display"] for spec in MODEL_SPECS.values()],
    "scientific_boundary": {
        "new_threshold_selected": False,
        "recalibration_performed": False,
        "model_fitting_performed": False,
        "score_modified": False,
        "outcome_modified": False,
        "cohort_modified": False,
        "experiment_2_started": False,
    },
}

write_json(PATHS["design"], design_payload)
write_csv(PATHS["point_estimates"], point_estimates)
write_parquet(PATHS["replicates"], bootstrap_replicates)
write_csv(PATHS["model_intervals"], model_intervals)
write_csv(PATHS["paired_comparisons"], paired_comparisons)
write_csv(PATHS["count_qc"], count_qc)
write_csv(PATHS["historical_concordance"], historical_concordance)

primary_artifact_keys = [
    "design",
    "point_estimates",
    "replicates",
    "model_intervals",
    "paired_comparisons",
    "count_qc",
    "historical_concordance",
]
for key in primary_artifact_keys:
    write_sidecar(PATHS[key])

readback = {
    "design": json.loads(PATHS["design"].read_text(encoding="utf-8"))["cell_id"] == "6C-4K0A",
    "point_estimates": len(pd.read_csv(PATHS["point_estimates"])) == 2,
    "replicates": pd.read_parquet(PATHS["replicates"]).shape == (2000, 45),
    "model_intervals": len(pd.read_csv(PATHS["model_intervals"])) == 34,
    "paired_comparisons": len(pd.read_csv(PATHS["paired_comparisons"])) == 8,
    "count_qc": len(pd.read_csv(PATHS["count_qc"])) == 2,
    "historical_concordance": pd.read_csv(PATHS["historical_concordance"])["value_reproduced_at_recorded_precision"].astype(bool).all(),
}
if not all(readback.values()):
    raise RuntimeError(f"Fresh table readback failed: {readback}")

for key in primary_artifact_keys:
    if not sidecar_ok(PATHS[key]):
        raise RuntimeError(f"Fresh sidecar verification failed for {PATHS[key]}")

qc_payload = {
    "cell_id": "6C-4K0A",
    "package_name": PACKAGE_NAME,
    "created_utc": CREATED_UTC,
    "analysis": "frozen_threshold_bootstrap_support_materialization",
    "checks_total": len(checks),
    "checks_passed": int(sum(item["passed"] for item in checks)),
    "checks_failed": int(sum(not item["passed"] for item in checks)),
    "checks": checks,
    "fresh_readback": readback,
    "bootstrap_elapsed_seconds": bootstrap_elapsed_seconds,
    "source_hashes": {
        "stage6b_evaluable": sha256_file(EVALUABLE_PARQUET),
        "prior_4j_manifest": sha256_file(PRIOR_MANIFEST),
    },
    "scientific_conclusions": {
        "full_ges_threshold_operating_performance": "effectively null",
        "no_star_threshold_operating_subset": "enriched but very low recall",
        "continuous_score_results_superseded": False,
        "clinical_threshold_claim_supported": False,
    },
    "experiment_2_started": False,
    "decision": (
        "PASS_STAGE6C_FROZEN_THRESHOLD_BOOTSTRAP_SUPPORT_MATERIALIZED_"
        "CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(PATHS["qc"], qc_payload)
write_sidecar(PATHS["qc"])

manifest_artifacts = []
for key in primary_artifact_keys + ["qc"]:
    path = PATHS[key]
    sc = sidecar_path(path)
    manifest_artifacts.append(
        {
            "artifact_key": key,
            "path": str(path),
            "relative_to_project": str(path.relative_to(PROJECT_DIR)),
            "file_name": path.name,
            "bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
            "sidecar_path": str(sc),
            "sidecar_sha256": sha256_file(sc),
        }
    )

manifest_payload = {
    "schema_version": "1.0",
    "cell_id": "6C-4K0A",
    "package_name": PACKAGE_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": CREATED_UTC,
    "analysis": "frozen_threshold_bootstrap_support_materialization",
    "immutable_sources": {
        "stage6b_evaluable": {
            "path": str(EVALUABLE_PARQUET),
            "sha256": EXPECTED_EVALUABLE_SHA256,
        },
        "prior_4j_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": EXPECTED_PRIOR_MANIFEST_SHA256,
        },
    },
    "artifacts": manifest_artifacts,
    "artifact_count": len(manifest_artifacts),
    "scientific_boundary": design_payload["scientific_boundary"],
    "experiment_2_started": False,
    "experiment_2_authorized": False,
    "decision": qc_payload["decision"],
}
write_json(PATHS["manifest"], manifest_payload)
write_sidecar(PATHS["manifest"])

fresh_manifest = json.loads(PATHS["manifest"].read_text(encoding="utf-8"))
if fresh_manifest.get("cell_id") != "6C-4K0A":
    raise RuntimeError("Fresh manifest identity verification failed.")
if fresh_manifest.get("decision") != qc_payload["decision"]:
    raise RuntimeError("Fresh manifest decision verification failed.")
if fresh_manifest.get("experiment_2_started") is not False:
    raise RuntimeError("Experiment 2 boundary failed in fresh manifest.")

for entry in fresh_manifest["artifacts"]:
    artifact_path = Path(entry["path"])
    if sha256_file(artifact_path) != entry["sha256"] or not sidecar_ok(artifact_path):
        raise RuntimeError(f"Fresh manifest artifact verification failed: {artifact_path}")

# Reverify immutable inputs after all writes.
if sha256_file(EVALUABLE_PARQUET) != EXPECTED_EVALUABLE_SHA256 or not sidecar_ok(EVALUABLE_PARQUET):
    raise RuntimeError("Stage 6B source changed during materialization.")
if sha256_file(PRIOR_MANIFEST) != EXPECTED_PRIOR_MANIFEST_SHA256 or not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Cell 6C-4J0 manifest changed during materialization.")


# --------------------------------------------------------------------------------------------------
# 9. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 165
print("\n" + separator)
print("STAGE 6C — CELL 6C-4K0A — FROZEN 0.50-THRESHOLD BOOTSTRAP SUPPORT MATERIALIZATION")
print(separator)
print(f"Notebook file name                  : {NOTEBOOK_FILENAME}")
print(f"Stage 6B source SHA-256             : PASS ({sha256_file(EVALUABLE_PARQUET)})")
print(f"Prior Cell 6C-4J0 manifest SHA-256  : PASS ({sha256_file(PRIOR_MANIFEST)})")
print(f"Frozen cohort                       : PASS ({EXPECTED_ROWS:,} rows; {EXPECTED_EVENTS:,} events; {EXPECTED_NEGATIVES:,} negatives)")
print(f"Frozen threshold                    : PASS ({THRESHOLD_RULE})")
print(f"Bootstrap design                    : PASS ({BOOTSTRAP_REPLICATES:,} paired row replicates; seed {BOOTSTRAP_SEED}; batch {BOOTSTRAP_BATCH_SIZE})")
print(f"Raw replicate table                 : PASS ({bootstrap_replicates.shape[0]:,} × {bootstrap_replicates.shape[1]})")
print(f"Model interval rows                 : PASS ({len(model_intervals)})")
print(f"Paired comparison rows              : PASS ({len(paired_comparisons)})")
print(f"Historical concordance              : PASS ({len(historical_concordance)}/{len(historical_concordance)})")
print(f"Fresh QC                            : PASS ({qc_payload['checks_passed']}/{qc_payload['checks_total']})")
print(f"Output table directory              : {TABLE_DIR}")
print(f"QC path                             : {PATHS['qc']}")
print(f"Manifest path                       : {PATHS['manifest']}")
print(f"Manifest SHA-256                    : PASS ({sha256_file(PATHS['manifest'])})")

print("\nKEY FROZEN-THRESHOLD MODEL INTERVALS")
print("-" * 165)
key_metrics = [
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "balanced_accuracy",
    "matthews_correlation",
    "relative_event_risk",
]
print(
    model_intervals.loc[
        model_intervals["metric"].isin(key_metrics),
        [
            "model",
            "metric",
            "point_estimate",
            "percentile_95_ci_lower",
            "percentile_95_ci_upper",
        ],
    ].to_string(index=False)
)

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print("-" * 165)
print(
    "The unchanged Full-GES 0.50 threshold remains effectively null: its balanced-accuracy, "
    "MCC, event-rate-difference, relative-risk, and enrichment intervals include their nulls."
)
print(
    "The unchanged No-star 0.50 threshold identifies an enriched subset, but its sensitivity "
    "remains only about 5%-6%; this does not reverse the negative continuous-score ablation."
)
print(
    "This cell only materializes the already completed Cell 6C-2C2 inference. It performs no "
    "new threshold selection, optimization, recalibration, model fitting, or Experiment 2 work."
)

print("\nCELL DECISION")
print("-" * 165)
print(qc_payload["decision"])
print(
    "The previously in-memory frozen-threshold bootstrap is now independently serialized, "
    "checksum-protected, historically concordant, and freshly reverified."
)
print(
    "Next authorized action: rerun Cell 6C-4K0 FINAL_CORRECTED_V2 from the beginning. "
    "Experiment 2 remains unstarted and unauthorized."
)


Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4K0A_Frozen_Threshold_Bootstrap_Support_Materialization.ipynb

Reconstructing frozen threshold bootstrap across 66,636 rows: 6,485 events and 60,151 negatives
  Completed 250/2,000 paired replicates (0.4s elapsed)
  Completed 500/2,000 paired replicates (0.8s elapsed)
  Completed 750/2,000 paired replicates (1.2s elapsed)
  Completed 1,000/2,000 paired replicates (1.5s elapsed)
  Completed 1,250/2,000 paired replicates (1.9s elapsed)
  Completed 1,500/2,000 paired replicates (2.2s elapsed)
  Completed 1,750/2,000 paired replicates (2.6s elapsed)
  Completed 2,000/2,000 paired replicates (3.0s elapsed)

STAGE 6C — CELL 6C-4K0A — FROZEN 0.50-THRESHOLD BOOTSTRAP SUPPORT MATERIALIZATION
Notebook file name                  : GES_Stage6C_Cell_6C_4K0A_Frozen_Threshold_Bootstrap_Support_Materialization.ipynb
Stage 6B source SHA-256             : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e03